In [ ]:
%pip install -r requirements.txt

In [ ]:
from utils.general import check_img_size, check_requirements, check_imshow, non_max_suppression, apply_classifier, \
     scale_coords, xyxy2xywh, strip_optimizer, set_logging, increment_path
from utils.torch_utils import select_device, load_classifier, time_synchronized
from utils.datasets import LoadStreams, LoadImages
from deep_sort.utils.parser import get_config
from models.experimental import attempt_load
from deep_sort.deep_sort import DeepSort
from moviepy.editor import VideoFileClip
from src.overlay import draw_target_box, draw_info_panel
from pathlib import Path
from numpy import random
import torch.backends.cudnn as cudnn
import moviepy.editor as mp
import pandas as pd
import numpy as np
import argparse
import torch
import time
import math
import cv2
import sys
import csv
import os

# Lane Detection

In [ ]:
# Grayscale conversion
def grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


# Gaussian blur
def gaussian_blur(img, kernel_size):
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)


# Canny edge detection
def canny(img, low_threshold, high_threshold):
    return cv2.Canny(img, low_threshold, high_threshold)


# Road area mask (HSV: gray road surface + yellow markings)
def get_lane_mask(img):
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    color_dict_HSV = {'black': [(0, 0, 0), (180, 255, 70)],
                      'white': [(0, 0, 231), (180, 18, 255)],
                      'yellow': [(25, 50, 70), (35, 255, 255)],
                      'gray': [(0, 0, 40), (180, 40, 230)],
                      'brown': [(20, 0, 50), (180, 125, 200)]}
    mask_yellow = cv2.inRange(hsv, *color_dict_HSV['yellow'])
    mask_gray = cv2.inRange(hsv, *color_dict_HSV['gray'])
    mask_lane = cv2.bitwise_or(mask_gray, mask_yellow)
    mask_lane = cv2.morphologyEx(mask_lane, cv2.MORPH_OPEN, kernel=np.ones((5, 5), dtype=np.uint8), iterations=6)
    mask_lane = cv2.morphologyEx(mask_lane, cv2.MORPH_CLOSE, kernel=np.ones((20, 20), dtype=np.uint8))
    contours, hierarchy = cv2.findContours(mask_lane, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        hull = cv2.convexHull(cnt)
        cv2.drawContours(mask_lane, [hull], 0, (255), -1)
    mask_lane = cv2.morphologyEx(mask_lane, cv2.MORPH_ERODE, kernel=np.ones((5, 5), dtype=np.uint8))
    return mask_lane


# Ratio of background pixels in the mask
def get_mask_ratio(mask):
    pixel_count = np.prod(mask.shape)
    zero_count = np.count_nonzero(mask == 0)
    return zero_count / pixel_count


# Region of interest
def region_of_interest(img, vertices):
    mask = np.zeros_like(img)
    if len(img.shape) > 2:
        channel_count = img.shape[2]
        ignore_mask_color = (255,) * channel_count
    else:
        ignore_mask_color = 255

    cv2.fillPoly(mask, vertices, ignore_mask_color)
    masked_image = cv2.bitwise_and(img, mask)
    return masked_image


# Hough transform
def hough_lines(img, rho, theta, threshold, min_line_len, max_line_gap):
    lines = cv2.HoughLinesP(img, rho, theta, threshold, np.array([]), minLineLength=min_line_len, maxLineGap=max_line_gap)

    min_slope_thr = 0.3
    max_slope_thr = 3
    if lines is None:
        lines = []
    else:
        lines = lines.reshape(-1, 4)
        lines = choose_lines(lines, min_slope_thr, max_slope_thr)

    line_img = np.zeros((img.shape[0], img.shape[1], 3), dtype=np.uint8)
    line_img, left_lane_line, right_lane_line = draw_lines(line_img, lines)
    return line_img, left_lane_line, right_lane_line


# Filter lane line candidates
def choose_lines(lines, min_slope_thr, max_slope_thr):
    abs_slopes = np.abs([get_slope(*line) for line in lines])
    chosen_indices = np.logical_and(min_slope_thr <= abs_slopes, abs_slopes <= max_slope_thr)
    lines = lines[chosen_indices]
    return lines


# Draw line segments
def draw_lines(img, lines, thickness=3):
    global last_left_line, last_right_line
    img_height, img_width, _ = img.shape

    slopes = np.array(list(map(lambda x: get_slope(*x), lines)))

    #left lines
    left_lines = lines[slopes < 0] if np.any(slopes) else []
    left_average_line, left_lines = get_average_line(left_lines, last_left_line)
    last_left_line = left_average_line
    #for line in left_lines:
        #cv2.line(img, line[:2], line[2:], [0, 255, 255], thickness)

    #right lines
    right_lines = lines[slopes > 0] if np.any(slopes) else []
    right_average_line, right_lines = get_average_line(right_lines, last_right_line)
    last_right_line = right_average_line
    #for line in right_lines:
        #cv2.line(img, line[:2], line[2:], [0, 255, 255], thickness)

    #draw lanes
    left_lane_line, right_lane_line = get_complete_lines(left_average_line, right_average_line, img_height, img_width)
    cv2.line(img, left_lane_line[:2], left_lane_line[2:], [70, 130, 180], thickness * 2)
    cv2.line(img, right_lane_line[:2], right_lane_line[2:], [70, 130, 180], thickness * 2)
    return img, left_lane_line, right_lane_line


# Average left and right lane lines
def get_average_line(lines, default_line):
    lines = np.array(lines)
    if not lines.any():
        return default_line, [default_line]

    n = 3
    normal_indices = []
    lines = np.concatenate([lines, [default_line] * n], axis=0) if np.any(default_line) else lines
    lines_mean = np.mean(lines, axis=0)
    lines_std = np.std(lines, axis=0)
    for m in range(1, 3):
        normal_indices = np.all(abs(lines - lines_mean) <= m * lines_std, axis=1)
        if np.any(normal_indices):
            break
    lines = lines[normal_indices]
    average_line = np.mean(lines, axis=0, dtype=np.int32) if np.any(lines) else default_line
    return average_line, lines


# Compute slope
def get_slope(x1, y1, x2, y2):
    if x2 == x1:
        return float('inf')
    else:
        return (y2 - y1) / (x2 - x1)


# Extend to full lane lines
def get_complete_lines(left_line_seg, right_line_seg, img_height, img_width):
    bottom_line = [0, img_height, img_width, img_height]
    left_line = [0, 0, 0, img_height]
    right_line = [img_width, 0, img_width, img_height]

    left_bottom_line = line_intersection(bottom_line, left_line_seg)
    if left_bottom_line[0] < 0:
        left_bottom_line = line_intersection(left_line, left_line_seg)

    right_bottom_line = line_intersection(bottom_line, right_line_seg)
    if right_bottom_line[0] >= img_width:
        right_bottom_line = line_intersection(right_line, right_line_seg)

    lanes_intersection = line_intersection(left_line_seg, right_line_seg)

    left_lane_line = left_bottom_line + lanes_intersection
    right_lane_line = lanes_intersection + right_bottom_line
    return left_lane_line, right_lane_line


# Compute intersection point
def line_intersection(line1, line2):
    xdiff = (line1[0] - line1[2], line2[0] - line2[2])
    ydiff = (line1[1] - line1[3], line2[1] - line2[3])

    def det(a, b):
        return a[0] * b[1] - a[1] * b[0]

    div = det(xdiff, ydiff)
    if div == 0:
        raise Exception('lines do not intersect')

    d = (det(line1[:2], line1[2:]), det(line2[:2], line2[2:]))
    x = det(d, xdiff) / div
    y = det(d, ydiff) / div
    return [round(x), round(y)]


# Overlay lanes on the original frame
def weighted_img(img, initial_img, α=0.8, β=0.5, γ=0.):
    return cv2.addWeighted(initial_img, α, img, β, γ)


def coordinates(file_path, frame_number, left_top, left_bottom, right_top, right_bottom):
    with open(file_path, mode='a', newline='') as file:
        writer = csv.writer(file)
        if file.tell() == 0:
            writer.writerow(["Frame", "Left_Top", "Left_Bottom", "Right_Top", "Right_Bottom"])
        writer.writerow([
            frame_number,
            f"({left_top[0]}, {left_top[1]})",
            f"({left_bottom[0]}, {left_bottom[1]})",
            f"({right_top[0]}, {right_top[1]})",
            f"({right_bottom[0]}, {right_bottom[1]})"])


def process_image(img, frame_number, output_file):
    gray_image = grayscale(img)
    gaus_blur = gaussian_blur(gray_image, 11)
    edges = canny(gaus_blur, 30, 120)

    imshape = img.shape
    mask_lane = get_lane_mask(img.copy())
    mask_ratio = get_mask_ratio(mask_lane)
    vertices = np.array([[(0, imshape[0] * 5 / 6),
                            (imshape[1] / 2, imshape[0] * 3 / 7),
                            (imshape[1], imshape[0] * 5 / 6)]], dtype=np.int32)
    edge_lane = region_of_interest(edges, vertices)

    if mask_ratio < 0.85:
        edge_lane = cv2.bitwise_and(edge_lane, edge_lane, mask=mask_lane)

    line_img, left_lane_line, right_lane_line = hough_lines(edge_lane, rho=1, theta=np.pi/90, threshold=30, min_line_len=20, max_line_gap=imshape[0] / 3)
    result = weighted_img(line_img, img)

    left_top = (left_lane_line[2], left_lane_line[3])
    left_bottom = (left_lane_line[0], left_lane_line[1])
    right_top = (right_lane_line[0], right_lane_line[1])
    right_bottom = (right_lane_line[2], right_lane_line[3])

    coordinates(output_file, frame_number, left_top, left_bottom, right_top, right_bottom)
    return result, left_lane_line, right_lane_line


def process_frame(frame):
    global frame_number
    frame_number += 1
    result, _, _ = process_image(frame, frame_number, output_file)
    return result


video_input = 'videos/CAR.mp4'
video_stem = Path(video_input).stem
video_output = f'outputs/{video_stem}_lanes.mp4'
output_file = f'outputs/{video_stem}_coordinates.csv'
os.makedirs('outputs', exist_ok=True)

frame_number = 0
with open(output_file, 'w') as f:
    pass

last_left_line = []
last_right_line = []

clip = VideoFileClip(video_input)
processed_clip = clip.fl_image(process_frame)
processed_clip.write_videofile(video_output, audio=False)

# Collision Warning

In [ ]:
foc = 800
real_hight_bicycle = 26
real_hight_car = 63
real_hight_motorcycle = 50
real_hight_bus = 128
real_hight_truck = 130


def detect_distance_car(h):
    dis_inch = (real_hight_car * foc) / h
    dis_cm = dis_inch * 2.54
    dis_cm = int(dis_cm)
    dis_m = dis_cm/100
    return dis_m

def detect_distance_bicycle(h):
    dis_inch = (real_hight_bicycle * foc) / h
    dis_cm = dis_inch * 2.54
    dis_cm = int(dis_cm)
    dis_m = dis_cm / 100
    return dis_m

def detect_distance_motorcycle(h):
    dis_inch = (real_hight_motorcycle * foc) / h
    dis_cm = dis_inch * 2.54
    dis_cm = int(dis_cm)
    dis_m = dis_cm/100
    return dis_m

def detect_distance_bus(h):
    dis_inch = (real_hight_bus * foc) / h
    dis_cm = dis_inch * 2.54
    dis_cm = int(dis_cm)
    dis_m = dis_cm/100
    return dis_m

def detect_distance_truck(h):
    dis_inch = (real_hight_truck * foc) / h
    dis_cm = dis_inch * 2.54
    dis_cm = int(dis_cm)
    dis_m = dis_cm/100
    return dis_m


# Match a detection box to the DeepSORT track with the highest IoU and return its track ID
def match_track(x, outputs, min_iou=0.3):
    if len(outputs) == 0:
        return None
    x1, y1, x2, y2 = [float(v) for v in x]
    best_id, best_iou = None, min_iou
    for o in outputs:
        ox1, oy1, ox2, oy2 = [float(v) for v in o[:4]]
        iw = max(0, min(x2, ox2) - max(x1, ox1))
        ih = max(0, min(y2, oy2) - max(y1, oy1))
        inter = iw * ih
        union = (x2 - x1) * (y2 - y1) + (ox2 - ox1) * (oy2 - oy1) - inter
        iou = inter / union if union > 0 else 0
        if iou > best_iou:
            best_iou, best_id = iou, int(o[4])
    return best_id


# Compute distance, relative speed, time-to-collision (TTC), and risk level for one vehicle
def analyze_target(x, speed, outputs, time_person, name=None, conf=None):
    h = int(x[3]) - int(x[1])
    dis_m = 1.00
    if name == 'bicycle':
        dis_m = detect_distance_bicycle(h)
    elif name == 'car':
        dis_m = detect_distance_car(h)
    elif name == 'motorcycle':
        dis_m = detect_distance_motorcycle(h)
    elif name == 'bus':
        dis_m = detect_distance_bus(h)
    elif name == 'truck':
        dis_m = detect_distance_truck(h)

    track_id = match_track(x, outputs)
    spd = None
    for sp in speed:
        if track_id is not None and sp[1] == track_id:
            spd = float(sp[0])
            break

    ttc, flag = None, 'No risk'
    if spd:
        ttc = dis_m / (spd * 0.277777778)    # convert km/h to m/s; TTC = distance / speed
        if ttc <= time_person * 0.5:         # high risk if TTC is below half of the reaction-and-braking time
            flag = 'High risk'

    return {'name': name, 'conf': conf, 'distance': dis_m, 'speed': spd,
            'track_id': track_id, 'ttc': ttc, 'flag': flag}


# Decide whether to trigger the warning sound
def trigger_sound(flag):
    if flag == "High risk":
      return True
    else:
      return False


# Insert the warning sound at the given timestamps
def insert_audio(video, audio_clip, timestamps):
    audio_clips = [video.audio] if video.audio else []

    for t in timestamps:
      if t + audio_clip.duration > video.duration:
          print(f"Warning: audio at {t} s exceeds the video length and was skipped.")
          continue
      audio_at_time = audio_clip.set_start(t).set_duration(audio_clip.duration)
      audio_clips.append(audio_at_time)

    final_audio = mp.CompositeAudioClip(audio_clips)
    return final_audio

# Vehicle Detection, Tracking, and Warning

In [ ]:
np.int = int
np.float = float
palette = (2 ** 11 - 1, 2 ** 15 - 1, 2 ** 20 - 1)


def attempt_load_custom(weights, map_location=None):
    checkpoint = torch.load(weights, map_location=map_location, weights_only=False)
    model = checkpoint['model'].float().eval()
    return model


def parse_coordinates(coord_str):
    return tuple(map(int, coord_str.strip("()").split(",")))


# Compute bounding box coordinates
def bbox_rel(image_width, image_height, *xyxy):
    bbox_left = min([xyxy[0].item(), xyxy[2].item()])
    bbox_top = min([xyxy[1].item(), xyxy[3].item()])
    bbox_w = abs(xyxy[0].item() - xyxy[2].item())
    bbox_h = abs(xyxy[1].item() - xyxy[3].item())
    x_c = (bbox_left + bbox_w / 2)
    y_c = (bbox_top + bbox_h / 2)
    w = bbox_w
    h = bbox_h
    return x_c, y_c, w, h


# Speed estimation
def Estimated_speed(locations, fps, width):
    present_IDs = []
    prev_IDs = []
    work_IDs = []
    work_IDs_index = []
    work_IDs_prev_index = []
    work_locations = []
    work_prev_locations = []
    speed = []
    for i in range(len(locations[1])):
        present_IDs.append(locations[1][i][2])
    for i in range(len(locations[0])):
        prev_IDs.append(locations[0][i][2])
    for m, n in enumerate(present_IDs):
        if n in prev_IDs:
            work_IDs.append(n)
            work_IDs_index.append(m)
    for x in work_IDs_index:
        work_locations.append(locations[1][x])
    for y, z in enumerate(prev_IDs):
        if z in work_IDs:
            work_IDs_prev_index.append(y)
    for x in work_IDs_prev_index:
        work_prev_locations.append(locations[0][x])
    for i in range(len(work_IDs)):
        speed.append(
            math.sqrt((work_locations[i][0] - work_prev_locations[i][0]) ** 2 +
                      (work_locations[i][1] - work_prev_locations[i][1]) ** 2) *
            width[work_locations[i][3]] / (work_locations[i][4]) * fps / 3 * 3.6 * 1.9)
    for i in range(len(speed)):
        speed[i] = [round(speed[i], 1), work_locations[i][2]]
    return speed


# Keep only vehicles in the same lane
def filter_detections(detections, vertices):
    polygon = np.array(vertices, dtype=np.int32)
    filtered_detections = []
    for det in detections:
        # Bottom-center point of the bounding box
        x_bottom_center = (det[0] + det[2]) / 2
        y_bottom_center = det[3]
        bottom_center = (float(x_bottom_center), float(y_bottom_center))
        # Check whether the bottom-center point is inside the lane polygon
        if cv2.pointPolygonTest(polygon, bottom_center, False) >= 0:
            filtered_detections.append(det)
    return filtered_detections


def detect(save_img=False):
    source, weights, view_img, save_txt, imgsz = opt.source, opt.weights, opt.view_img, opt.save_txt, opt.img_size
    save_img = not opt.nosave and not source.endswith('.txt')
    webcam = source.isnumeric() or source.endswith('.txt') or source.lower().startswith(
        ('rtsp://', 'rtmp://', 'http://', 'https://'))

    save_dir = Path(increment_path(Path(opt.project) / opt.name, exist_ok=opt.exist_ok))
    (save_dir / 'labels' if save_txt else save_dir).mkdir(parents=True, exist_ok=True)

    csv_path = f"outputs/{Path(source).stem}_coordinates.csv"
    lane_data = pd.read_csv(csv_path)

    lane_data['Left_Top'] = lane_data['Left_Top'].apply(parse_coordinates)
    lane_data['Left_Bottom'] = lane_data['Left_Bottom'].apply(parse_coordinates)
    lane_data['Right_Top'] = lane_data['Right_Top'].apply(parse_coordinates)
    lane_data['Right_Bottom'] = lane_data['Right_Bottom'].apply(parse_coordinates)


    capture = cv2.VideoCapture(source)
    fps = capture.get(cv2.CAP_PROP_FPS)
    audio_frames = []
    frame_fature = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Initialize DeepSORT
    cfg = get_config()
    cfg.merge_from_file(opt.config_deepsort)
    deepsort = DeepSort(cfg.DEEPSORT.REID_CKPT,
                        max_dist=cfg.DEEPSORT.MAX_DIST, min_confidence=cfg.DEEPSORT.MIN_CONFIDENCE,
                        nms_max_overlap=cfg.DEEPSORT.NMS_MAX_OVERLAP, max_iou_distance=cfg.DEEPSORT.MAX_IOU_DISTANCE,
                        max_age=cfg.DEEPSORT.MAX_AGE, n_init=cfg.DEEPSORT.N_INIT, nn_budget=cfg.DEEPSORT.NN_BUDGET,
                        use_cuda=True)

    # Vehicle widths by class and the reaction-and-braking time (seconds)
    width = [0, 0.2, 1.85, 0.5, 0, 2.3, 0, 2.5]  # real widths (m) indexed by COCO class: bicycle, car, motorcycle, bus, truck
    time_person = 3
    locations = []
    speed = []

    set_logging()
    device = select_device(opt.device)
    half = device.type != 'cpu'

    # Load model
    model = attempt_load_custom(weights, map_location=device)
    stride = int(model.stride.max())
    imgsz = check_img_size(imgsz, s=stride)
    if half:
        model.half()

    # Second-stage classifier
    classify = False
    if classify:
        modelc = load_classifier(name='resnet101', n=2)
        modelc.load_state_dict(torch.load('weights/resnet101.pt', map_location=device)['model']).to(device).eval()

    # Set Dataloader
    vid_path, vid_writer = None, None
    if webcam:
        view_img = check_imshow()
        cudnn.benchmark = True
        dataset = LoadStreams(source, img_size=imgsz, stride=stride)
    else:
        dataset = LoadImages(source, img_size=imgsz, stride=stride)

    names = model.module.names if hasattr(model, 'module') else model.names



    # Run inference
    if device.type != 'cpu':
        model(torch.zeros(1, 3, imgsz, imgsz).to(device).type_as(next(model.parameters())))
    t0 = time.time()
    for frame_idx, (path, img, im0s, vid_cap) in enumerate(dataset):
        img = torch.from_numpy(img).to(device)
        img = img.half() if half else img.float()
        img /= 255.0
        if img.ndimension() == 3:
            img = img.unsqueeze(0)

        # Inference
        t1 = time_synchronized()
        pred = model(img, augment=opt.augment)[0]

        # Apply NMS
        pred = non_max_suppression(pred, opt.conf_thres, opt.iou_thres, classes=opt.classes, agnostic=opt.agnostic_nms)
        t2 = time_synchronized()
        # Lane polygon vertices for the current frame
        lane_vertices = [
            lane_data['Left_Top'][frame_idx],
            lane_data['Left_Bottom'][frame_idx],
            lane_data['Right_Bottom'][frame_idx],
            lane_data['Right_Top'][frame_idx] ]

        # Apply Classifier
        if classify:
            pred = apply_classifier(pred, modelc, img, im0s)

        # Process detections
        for i, det in enumerate(pred):
            if webcam:
                p, s, im0, frame = path[i], '%g: ' % i, im0s[i].copy(), dataset.count
            else:
                p, s, im0, frame = path, '', im0s, getattr(dataset, 'frame', 0)

            p = Path(p)
            save_path = str(save_dir / "output.mp4")
            txt_path = str(save_dir / 'labels' / p.stem) + ('' if dataset.mode == 'image' else f'_{frame}')
            s += '%gx%g ' % img.shape[2:]
            gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]
            targets = []   # in-lane vehicles in this frame, used by the info panel

            # Draw lane lines
            im0 = cv2.line(im0, lane_data['Left_Top'][frame_idx], lane_data['Left_Bottom'][frame_idx], (180, 130, 70), 5)
            im0 = cv2.line(im0, lane_data['Right_Top'][frame_idx], lane_data['Right_Bottom'][frame_idx], (180, 130, 70), 5)

            # Filter separately so an empty result does not break the steps below
            if len(det):
                # Rescale boxes from img_size to im0 size
                det[:, :4] = scale_coords(img.shape[2:], det[:, :4], im0.shape).round()
                # Boxes must be rescaled to the original frame size before lane filtering
                det = filter_detections(det, lane_vertices)

            if len(det):
                # Speed estimation
                bbox_xywh = []
                confs = []
                classes = []
                img_h, img_w, _ = im0.shape
                # Convert detections to the DeepSORT input format
                for *xyxy, conf, cls in det:
                    x_c, y_c, bbox_w, bbox_h = bbox_rel(img_w, img_h, *xyxy)
                    obj = [x_c, y_c, bbox_w, bbox_h]
                    bbox_xywh.append(obj)
                    confs.append([conf.item()])
                    classes.append([cls.item()])
                xywhs = torch.Tensor(bbox_xywh)
                confss = torch.Tensor(confs)
                classes = torch.Tensor(classes)
                # Update DeepSORT tracks
                outputs = deepsort.update(xywhs, confss, im0, classes)
                box_centers = []
                for i, each_box in enumerate(outputs):
                    if each_box[5] == 1 or each_box[5] == 2 or each_box[5] == 3 or each_box[5] == 5 or each_box[5] == 7:
                        box_centers.append([(each_box[0] + each_box[2]) / 2, (each_box[1] + each_box[3]) / 2, each_box[4],
                                        each_box[5], each_box[2] - each_box[0]])
                location = box_centers

                if len(location) != 0:
                    with open('outputs/track.txt', 'a+') as track_record:
                        track_record.write('frame:%s\n' % str(frame_idx))
                        for j in range(len(location)):
                            track_record.write('id:%s,x:%s,y:%s\n' % (str(location[j][2]), str(location[j][0]), str(location[j][1])))
                locations.append(location)
                # Estimate and log speed every 10 frames
                if len(locations) == 10:
                    if len(locations[0]) and len(locations[-1]) != 0:
                        locations = [locations[0], locations[-1]]
                        speed = Estimated_speed(locations, fps, width)
                    with open('outputs/speed.txt', 'a+') as speed_record:
                        for sp in speed:
                            speed_record.write('id:%s %skm/h\n' % (str(sp[1]), str(sp[0])))
                    locations = []

                # Write results
                for *xyxy, conf, cls in reversed(det):
                    conf2 = float(f'{conf:.2f}')
                    if conf2 > 0.40:   # skip boxes with confidence <= 0.4
                        # Only vehicles: bicycle, car, motorcycle, bus, truck
                        if names[int(cls)] == 'bicycle' or names[int(cls)] == 'car' or names[int(cls)] == 'motorcycle' or names[int(cls)] == 'bus' or names[int(cls)] == 'truck':
                            if save_txt:
                                xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(
                                    -1).tolist()
                                line = (cls, *xywh, conf) if opt.save_conf else (cls, *xywh)
                                with open(txt_path + '.txt', 'a') as f:
                                    f.write(('%g ' * len(line)).rstrip() % line + '\n')

                            if save_img or view_img:
                                # Compute distance, speed, TTC, and risk; record the frame for the warning sound if high risk
                                info = analyze_target(xyxy, speed, outputs, time_person, name=names[int(cls)], conf=float(conf))
                                if trigger_sound(info['flag']):
                                    audio_frames.append(frame_idx)
                                # Label above the box: class, distance, speed; green = safe, red = high risk
                                draw_target_box(im0, xyxy, info, line_thickness=3)
                                targets.append(info)

            # Draw the bottom-center point used for lane filtering
            for det_f in det:
              x_bottom_center = (det_f[0] + det_f[2]) / 2
              y_bottom_center = det_f[3]
              bottom_center = (int(x_bottom_center), int(y_bottom_center))
              im0 = cv2.circle(im0, bottom_center, 3, color=(255, 0, 255), thickness=-1)

            # Info panel in the top-left corner
            im0 = draw_info_panel(im0, targets, time_person)

            #  Print time (inference + NMS)
            print(f'{s}Done. ({t2 - t1:.3f}s)')

            if save_img:
                if dataset.mode == 'image':
                    cv2.imwrite(save_path, im0)
                else:
                    if vid_path != save_path:
                        vid_path = save_path
                        if isinstance(vid_writer, cv2.VideoWriter):
                            vid_writer.release()
                        if vid_cap:
                            fps = vid_cap.get(cv2.CAP_PROP_FPS)
                            w = int(vid_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                            h = int(vid_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                        else:
                            fps, w, h = 30, im0.shape[1], im0.shape[0]
                            save_path += '.mp4'
                        vid_writer = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
                    vid_writer.write(im0)
    if save_img:
      if dataset.mode != 'image':
        vid_writer.release()

    video_path = save_path
    audio_path = "assets/warning.mp3"

    try:
        video = mp.VideoFileClip(video_path)
        print(f"Video duration: {video.duration} s")
    except Exception as e:
        raise ValueError(f"Failed to load video: {e}")

    try:
        audio_warning = mp.AudioFileClip(audio_path)
        print(f"Audio duration: {audio_warning.duration} s")
    except Exception as e:
        raise ValueError(f"Failed to load audio: {e}")

    timestamps = frame_times = [frame / fps for frame in audio_frames]
    last_timestamp = 0
    filtered_timestamps = []
    for timestamp in timestamps:
      if (timestamp - last_timestamp) < 0.65:
        continue
      else:
        filtered_timestamps.append(timestamp)
        last_timestamp = timestamp

    # Add warning sounds to the video
    new_audio = insert_audio(video, audio_warning, filtered_timestamps)
    final_video = video.set_audio(new_audio)

    # Export the final video
    output_path = str(save_dir / "output_video.mp4")
    final_video.write_videofile(output_path, codec="libx264", audio_codec="aac")
    print(f"Video saved: {output_path}")

    # dictionary of timestamps
    temp_dict = {'timestamps': filtered_timestamps}
    df = pd.DataFrame(temp_dict)
    # saving the dataframe
    csv_save_path = str(save_dir / "timestamps.csv")
    df.to_csv(csv_save_path)

    if save_txt or save_img:
        s = f"\n{len(list(save_dir.glob('labels/*.txt')))} labels saved to {save_dir / 'labels'}" if save_txt else ''
        print(f"Results saved to {save_dir}{s}")

    print(f'Done. ({time.time() - t0:.3f}s)')



if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--weights', nargs='+', type=str, default='weights/yolov5s.pt', help='model.pt path(s)')
    parser.add_argument('--source', type=str, default='videos/CAR.mp4', help='source')
    parser.add_argument('--img-size', type=int, default=640, help='inference size (pixels)')
    parser.add_argument('--conf-thres', type=float, default=0.01, help='object confidence threshold')
    parser.add_argument('--iou-thres', type=float, default=0.01, help='IOU threshold for NMS')
    parser.add_argument('--device', default='', help='cuda device, i.e. 0 or 0,1,2,3 or cpu')
    parser.add_argument('--view-img', action='store_true', help='display results',default=True)
    parser.add_argument('--save-txt', action='store_true', help='save results to *.txt')
    parser.add_argument('--save-conf', action='store_true', help='save confidences in --save-txt labels')
    parser.add_argument('--nosave', action='store_true', help='do not save images/videos')
    parser.add_argument('--classes', nargs='+', type=int, help='filter by class: --class 0, or --class 0 2 3')
    parser.add_argument('--agnostic-nms', action='store_true', help='class-agnostic NMS')
    parser.add_argument('--augment', action='store_true', help='augmented inference')
    parser.add_argument('--update', action='store_true', help='update all models')
    parser.add_argument('--project', default='runs', help='save results to project/name')
    parser.add_argument('--name', default='detect', help='save results to project/name')
    parser.add_argument('--exist-ok', action='store_true', help='existing project/name ok, do not increment')
    parser.add_argument("--config_deepsort", type=str, default="deep_sort/configs/deep_sort.yaml")
    opt, _ = parser.parse_known_args()
    # check_requirements(exclude=('pycocotools', 'thop'))

    print(opt)
    # check_requirements(exclude=('pycocotools', 'thop'))

    with torch.no_grad():
        if opt.update:
            for opt.weights in ['yolov5s.pt', 'yolov5m.pt', 'yolov5l.pt', 'yolov5x.pt']:
                detect()
                strip_optimizer(opt.weights)
        else:
            detect()